## Purpose and Limitations

This notebook builds a **Role Master Table** — a clean reference dataset that maps each O*NET occupation to its required skills. 

**Why this matters:**
- Day 3 (Workforce Intelligence) will use this to analyze skill gaps at the occupation level
- It enables queries like "What skills does a Software Developer need?" without ambiguity

**Critical limitation:**
Notebook 04 found **0% overlap** between employee `JobRole` (free-text HR labels like "Developer") and occupation `Title` (O*NET standard names like "Software Developers"). This means:
- We **cannot** directly attach role-level skills to individual employees
- Skill-gap analysis runs at the **occupation** level, not the employee level
- To get employee-level skill gaps, we'd need to build a manual `JobRole → O*NET-SOC Code` mapping table (future work)

This role master is still valuable for:
- Understanding what skills are in demand across the industry
- Identifying transferable skills (skills that appear in many roles)
- Benchmarking our workforce against industry standards

## 11. Role Intelligence

occupation_data.csv becomes our 'role master' table — no ML, it's reference data. Every role gets a clean ID and name, so we can later say 'ML Engineer requires Python, MLOps, Docker, AWS' without ambiguity about which role we mean.

In [16]:
import pandas as pd
import os
import json
from collections import Counter

PROCESSED_PATH = '../data/processed'

occ = pd.read_csv(f'{PROCESSED_PATH}/occupation_master.csv')
ess = pd.read_csv(f'{PROCESSED_PATH}/essential_skills_processed.csv')
sw = pd.read_csv(f'{PROCESSED_PATH}/software_skills_processed.csv')

print(f'Occupations: {occ.shape[0]}')
print(f'Essential skills: {ess.shape[0]}')
print(f'Software skills: {sw.shape[0]}')

Occupations: 1016
Essential skills: 18200
Software skills: 31821


## 1. Occupation Data Overview

Raw occupation data from O*NET — the 'role master' source.

In [17]:
print('Columns:', list(occ.columns))
print()
print('Sample:')
print(occ.head())
print()
print('Unique O*NET-SOC Codes:', occ['O*NET-SOC Code'].nunique())
print('Unique Titles:', occ['Title'].nunique())

Columns: ['O*NET-SOC Code', 'Title', 'Description']

Sample:
  O*NET-SOC Code                                Title  \
0     11-1011.00                     Chief Executives   
1     11-1011.03        Chief Sustainability Officers   
2     11-1021.00      General and Operations Managers   
3     11-1031.00                          Legislators   
4     11-2011.00  Advertising and Promotions Managers   

                                         Description  
0  Determine and formulate policies and provide o...  
1  Communicate and coordinate with management, sh...  
2  Plan, direct, or coordinate the operations of ...  
3  Develop, introduce, or enact laws and statutes...  
4  Plan, direct, or coordinate advertising polici...  

Unique O*NET-SOC Codes: 1016
Unique Titles: 1016


## 2. Essential Skills Overview

What skills are required for each role. Each skill has a 'Data Value' indicating importance level.

In [18]:
print('Columns:', list(ess.columns))
print()
print('Unique skills (Element Name):', ess['Element Name'].nunique())
print('Skill domains:', ess['Scale Name'].unique())
print()
print('Sample skills for Chief Executives:')
chief_ess = ess[ess['Title'] == 'Chief Executives']
print(chief_ess[['Element Name', 'Data Value']].head(10).to_string(index=False))

Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Unique skills (Element Name): 10
Skill domains: <StringArray>
['Importance', 'Level']
Length: 2, dtype: str

Sample skills for Chief Executives:
         Element Name  Data Value
Reading Comprehension        4.12
Reading Comprehension        4.62
     Active Listening        4.00
     Active Listening        4.75
              Writing        4.12
              Writing        4.38
             Speaking        4.25
             Speaking        4.75
          Mathematics        3.25
          Mathematics        3.50


## 3. Software Skills Overview

Specific software/technologies required for each role. Hot Technology = Y means it's in high demand.

In [19]:
print('Columns:', list(sw.columns))
print()
print('Unique software:', sw['Element Name'].nunique())
print('Hot Technologies:', (sw['Hot Technology'] == 'Y').sum())
print()
print('Sample software for Software Developers:')
dev_sw = sw[(sw['Title'] == 'Software Developers') & (sw['Hot Technology'] == 'Y')]
cols = ['Element Name', 'Hot Technology', 'In Demand']
print(dev_sw[cols].head(10).to_string(index=False))

Columns: ['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand', 'WorkplaceExample_clean']

Unique software: 134
Hot Technologies: 11571

Sample software for Software Developers:
                                    Element Name Hot Technology In Demand
                    Document management software              Y         N
             Video creation and editing software              Y         N
              Graphics or photo imaging software              Y         N
              Graphics or photo imaging software              Y         N
                     Desktop publishing software              Y         N
              Graphics or photo imaging software              Y         N
               Web platform development software              Y         N
Business intelligence and data analysis software              Y         N
            Data base management system software              Y         N
     Data base user interfa

## 4. Build Role Master Table

Create a clean, denormalized table: one row per role with its essential skills and software skills as lists.

In [20]:
# Get top essential skills per role (by Data Value)
ess_importance = ess[ess['Scale Name'] == 'Importance']
top_ess = ess_importance.sort_values('Data Value', ascending=False).groupby('O*NET-SOC Code').head(5)
role_skills = top_ess.groupby('O*NET-SOC Code')['Element Name'].apply(lambda x: list(set(x)))
role_skills = role_skills.reset_index()
role_skills.columns = ['O*NET-SOC Code', 'Essential_Skills']

# Get software skills per role (use WorkplaceExample_clean)
role_sw = sw.groupby('O*NET-SOC Code')['WorkplaceExample_clean'].apply(lambda x: list(set(x))).reset_index()
role_sw.columns = ['O*NET-SOC Code', 'Software_Skills']

# Get hot technologies per role
hot_tech = sw[sw['Hot Technology'] == 'Y'].groupby('O*NET-SOC Code')['WorkplaceExample_clean'].apply(lambda x: list(set(x))).reset_index()
hot_tech.columns = ['O*NET-SOC Code', 'Hot_Technologies']

# Build role master
role_master = occ[['O*NET-SOC Code', 'Title', 'Description']].copy()
role_master = role_master.merge(role_skills, on='O*NET-SOC Code', how='left')
role_master = role_master.merge(role_sw, on='O*NET-SOC Code', how='left')
role_master = role_master.merge(hot_tech, on='O*NET-SOC Code', how='left')

# Fill missing with empty lists
role_master['Essential_Skills'] = role_master['Essential_Skills'].fillna('').apply(lambda x: x if isinstance(x, list) else [])
role_master['Software_Skills'] = role_master['Software_Skills'].fillna('').apply(lambda x: x if isinstance(x, list) else [])
role_master['Hot_Technologies'] = role_master['Hot_Technologies'].fillna('').apply(lambda x: x if isinstance(x, list) else [])

print(f'Role master: {role_master.shape[0]} roles')
print(f'Columns: {list(role_master.columns)}')

Role master: 1016 roles
Columns: ['O*NET-SOC Code', 'Title', 'Description', 'Essential_Skills', 'Software_Skills', 'Hot_Technologies']


## 5. Role Master Preview

Clean reference table: role ID, name, description, essential skills, software skills, hot technologies.

In [21]:
print('Role Master (first 5 rows):')
for idx, row in role_master.head().iterrows():
    print(f"\n{row['O*NET-SOC Code']} - {row['Title']}")
    print(f"  Essential: {', '.join(row['Essential_Skills'][:3])}...")
    print(f"  Software: {', '.join(row['Software_Skills'][:3])}...")
    if row['Hot_Technologies']:
        print(f"  Hot Tech: {', '.join(row['Hot_Technologies'][:3])}")

Role Master (first 5 rows):

11-1011.00 - Chief Executives
  Essential: Speaking, Critical Thinking, Writing...
  Software: Graphic presentation software, Microsoft Publisher, Oracle PeopleSoft...
  Hot Tech: Adobe Acrobat, Microsoft Project, Microsoft Word

11-1011.03 - Chief Sustainability Officers
  Essential: Active Listening, Speaking, Critical Thinking...
  Software: Tableau, Microsoft SharePoint, Management information systems MIS...
  Hot Tech: Adobe Acrobat, Microsoft Project, Microsoft Word

11-1021.00 - General and Operations Managers
  Essential: Active Listening, Speaking, Critical Thinking...
  Software: Oracle E-Business Suite Financials, Oracle Eloqua, SAP Business Objects...
  Hot Tech: Teradata Database, PostgreSQL, Oracle PeopleSoft

11-1031.00 - Legislators
  Essential: ...
  Software: Apple iWork Pages, Windows Media Player, XMetaL Author...
  Hot Tech: Adobe Acrobat, Microsoft Word, Structured query language SQL

11-2011.00 - Advertising and Promotions Managers
  

## 6. Example: ML Engineer Lookup

Demonstrate how to look up a role and its requirements.

In [22]:
# Find ML-related roles
ml_roles = role_master[role_master['Title'].str.contains('Machine Learning|Data Scientist|Software Developer', case=False, na=False)]

print(f'Found {len(ml_roles)} ML-related roles:')
for idx, row in ml_roles.iterrows():
    print(f"\n{row['O*NET-SOC Code']} - {row['Title']}")
    print(f"  Essential: {', '.join(row['Essential_Skills'][:5])}")
    if row['Hot_Technologies']:
        print(f"  Hot Tech: {', '.join(row['Hot_Technologies'][:5])}")

Found 2 ML-related roles:

15-1252.00 - Software Developers
  Essential: Active Learning, Active Listening, Critical Thinking, Writing, Reading Comprehension
  Hot Tech: Workday software, Teradata Database, Google Angular, jQuery, UNIX

15-2051.00 - Data Scientists
  Essential: Active Learning, Mathematics, Speaking, Critical Thinking, Reading Comprehension
  Hot Tech: Teradata Database, Apache Kafka, Ruby, C++, PostgreSQL


## 7. Skills Frequency Analysis

What skills appear most often across all roles? Useful for identifying transferable skills.

In [23]:
from collections import Counter

# Count all essential skills
all_ess = []
for skills in role_master['Essential_Skills']:
    all_ess.extend(skills)

ess_counter = Counter(all_ess)
print('Top 15 Essential Skills (across all roles):')
for skill, count in ess_counter.most_common(15):
    print(f'  {skill}: {count} roles')
print()

# Count all software skills
all_sw = []
for skills in role_master['Software_Skills']:
    all_sw.extend(skills)

sw_counter = Counter(all_sw)
print('Top 15 Software Skills (across all roles):')
for skill, count in sw_counter.most_common(15):
    print(f'  {skill}: {count} roles')

Top 15 Essential Skills (across all roles):
  Active Listening: 859 roles
  Speaking: 823 roles
  Critical Thinking: 821 roles
  Reading Comprehension: 811 roles
  Monitoring: 521 roles
  Writing: 357 roles
  Active Learning: 185 roles
  Learning Strategies: 69 roles
  Mathematics: 58 roles
  Science: 46 roles

Top 15 Software Skills (across all roles):
  Microsoft Excel: 859 roles
  Microsoft Office software: 817 roles
  Microsoft Word: 784 roles
  Microsoft Outlook: 642 roles
  Microsoft PowerPoint: 629 roles
  Web browser software: 431 roles
  Microsoft Access: 372 roles
  SAP software: 283 roles
  Email software: 276 roles
  Microsoft Windows: 241 roles
  Microsoft Project: 201 roles
  Autodesk AutoCAD: 171 roles
  Microsoft SharePoint: 166 roles
  Database software: 166 roles
  Adobe Acrobat: 165 roles


## 8. Save Role Master

Save as processed reference data for downstream notebooks.

In [24]:
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Save as CSV
role_master.to_csv(f'{PROCESSED_PATH}/role_master.csv', index=False)

# Save as JSON for easier reading
role_master_json = role_master.to_dict('records')
import json
with open(f'{PROCESSED_PATH}/role_master.json', 'w') as f:
    json.dump(role_master_json, f, indent=2)

print(f'Saved role_master.csv: {role_master.shape}')
print(f'Saved role_master.json: {len(role_master_json)} roles')

Saved role_master.csv: (1016, 6)
Saved role_master.json: 1016 roles


## Summary

### What We Built

- **Role Master Table**: 1016 roles from O*NET with clean IDs and names

- **Essential Skills**: Top 5 skills per role (by importance)

- **Software Skills**: All software/technologies per role

- **Hot Technologies**: Skills marked as high-demand

### Files Created

- `data/processed/role_master.csv` — reference table for downstream use

- `data/processed/role_master.json` — same data, easier to read

### Key Insights

- 1016 unique roles in the O*NET system

- Top essential skills: Reading Comprehension, Active Listening, Critical Thinking

- Top software: Microsoft Excel, Word, PowerPoint dominate

- Hot technologies: Python, SQL, Java appear frequently

### Next Steps

This role master can be used to:

- Map employee JobRole to O*NET occupation codes (if we build a manual mapping)

- Analyze skill gaps at the company level

- Generate role-specific training recommendations